Clone the SimPip repo:

In [1]:
!git clone https://$GITHUB_TOKEN@github.com/PrimozRavbar/SimPipeline.git

Cloning into 'SimPipeline'...
remote: Enumerating objects: 197, done.
remote: Counting objects: 100% (197/197), done.
remote: Compressing objects: 100% (145/145), done.
remote: Total 197 (delta 94), reused 141 (delta 48), pack-reused 0 (from 0)
Receiving objects: 100% (197/197), 8.14 MiB | 16.38 MiB/s, done.
Resolving deltas: 100% (94/94), done.


In [2]:
%cd /content/SimPipeline
!git pull

/content/SimPipeline
Already up to date.


Load the movielens data:

In [3]:
from world.external_data_loader import load_movielens

world_data = load_movielens("/content/SimPipeline/ml-100k")

print("Loader OK:", len(world_data["user_objects"]), len(world_data["movie_objects"]), len(world_data["rating_objects"]))

/content/SimPipeline/world/external_data_loader.py:27: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  ratings = pd.read_csv(


Loader OK: 943 1682 100000


In [4]:
import importlib

import ml_system.datalake
import ml_system.training_dataset
import ml_system.evaluation
import ml_system.build_recommendation_system

from ml_system.build_recommendation_system import build_recommendation_system
from ml_system.evaluation import RetrievalEvaluator, evaluate_popularity

Build a new system:

In [5]:
from ml_system.build_recommendation_system import build_recommendation_system

system = build_recommendation_system(
    movie_objects=world_data["movie_objects"],
    user_objects=world_data["user_objects"],
    user_states=world_data["user_states"],
    movie_states=world_data["movie_states"],
)

print("System build OK")

System build OK


Empty the datalake if need be:

In [ ]:
system["datalake"].events.clear()
print("DataLake events:", len(system["datalake"].events))

# Run the simulation:

In [6]:
system["sim"].run(5000)

Streaming output truncated to the last 5000 lines.
WatchEvent(timestamp=3814, user_id=577, movie_id=298, watch_time=85.8770115157477, completion=0.3569228235849402)
FINAL GENRE PREF 577 {'Action': 0.3333333333333333, 'Crime': 0.16666666666666666, 'Romance': 0.16666666666666666, 'Sci-Fi': 0.16666666666666666, 'Thriller': 0.16666666666666666}
World processing
User 885 clicked 794
RecommendationShownEvent(timestamp=3815, user_id=885, recommendation_id=3815, movie_ids=[240, 71, 1409, 449, 657, 595, 1219, 249, 709, 1502, 1033, 1411, 425, 189, 794, 298, 95, 411, 52, 274, 838, 414, 820, 1054, 276, 620, 1476, 856, 1132, 1293, 423, 1049, 436, 901, 69, 1588, 878, 1511, 1436, 367, 225, 1477, 556, 831, 309, 1138, 24, 1389, 141, 1313, 892, 1594, 432, 850, 215, 932, 408, 1516, 986, 746, 841, 951, 982, 217, 1092, 1500, 1291, 49, 428, 894, 343, 682, 228, 1421, 1254, 1210, 30, 495, 1562, 618, 464, 130, 1271, 1183, 1402, 41, 936, 1626, 861, 1451, 435, 763, 697, 1649, 1547, 869, 1515, 154, 1115, 974])
Re

In [7]:
events = system["datalake"].events

from collections import Counter

counts = Counter(type(event).__name__ for event in events)

print("Total events:", len(events))
print(counts)

Total events: 10679
Counter({'RecommendationShownEvent': 5000, 'RecommendationClickedEvent': 3014, 'WatchEvent': 2429, 'RatingEvent': 236})


Run the offline feature computation for training (Spark jobs...):

In [8]:
system["spark"].run_offline_feature_job(
    system["datalake"].events,
    system["offline_pipeline"].feature_pipeline
)
print("Spark offline feature job complete.")

Spark offline feature job complete.


# Train a Two-Tower model:

In [9]:
from ml_system.retrain import retrain

system["offline_pipeline"].dataset_generator.negative_samples = 5
system["offline_pipeline"].trainer.epochs = 5

artifacts = retrain(system)

epoch: 1 loss: 3.6147301197052
epoch: 2 loss: 3.5476998931483217
epoch: 3 loss: 3.451932900830319
epoch: 4 loss: 3.30827910900116
epoch: 5 loss: 3.1304550823412445


Test the training quality basic metrics (Recall@20):

In [10]:
from simulation.events import RecommendationClickedEvent, WatchEvent
from simulation.events import RetrievalRequest

events = system["datalake"].events

hits = 0
evaluated = 0

for event in events:
    if not isinstance(event, (RecommendationClickedEvent, WatchEvent)):
        continue

    recommendations = system["retriever"].retrieve(
        RetrievalRequest(
            user_id=event.user_id,
            k=20
        )
    )

    evaluated += 1

    if event.movie_id in recommendations:
        hits += 1

recall_at_20 = hits / evaluated if evaluated else 0.0

print(f"Hits: {hits}")
print(f"Evaluated: {evaluated}")
print(f"Recall@20: {recall_at_20:.4%}")

Hits: 719
Evaluated: 5443
Recall@20: 13.2096%


To improve Recall@20, train for more epochs over the same datalake.

Next, EMPTY the datalake and run another simulation (do NOT rebuild the System - keep the same model parameters).

Train with the new data in the datalake. The final Recall@20 should substentially increase (3X).